# Train the FiQA reranker on Google Colab

This notebook mirrors `scripts/train_reranker.py` and `configs/step3.yaml`. It fine-tunes the pinned MiniLM cross-encoder on the prepared Arrow pairs, selects a checkpoint using validation nDCG@10, then evaluates test once. It does not create data or change the official splits.

**Prerequisites:** a Colab runtime with a GPU, Google Drive access, and the project’s `configs/step3.yaml` plus `data/tokenized/fiqa_fast_minilm_l6_v2/` copied into `MyDrive/fiqa-reranker/`. Preserve the `train`, `validation`, and `test` folder names. Do not upload `.venv`, raw datasets, or local model checkpoints; this notebook fetches the pinned model itself.

Run top to bottom. Training is expensive; review `MAX_EPOCHS` and `BATCH_SIZE` in the config before executing the training cell. Outputs and checkpoints are written to Drive so they survive Colab disconnections.

## Steps
1. Install notebook dependencies and verify the selected GPU.
2. Mount Drive, load config/data, and validate labels and split sizes.
3. Download the pinned model, train with dynamic padding, and validate each epoch.
4. Restore the best validation checkpoint and evaluate test once.


## 1. Install pinned Python dependencies

Colab supplies a CUDA-enabled PyTorch build. We leave that installation intact and pin the data/model packages used by this notebook.

In [1]:
!nvidia-smi

Sun Oct 18 11:43:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%pip -q install datasets==5.1.0 transformers==5.20.1 pandas==3.0.6 pyyaml==6.0.3 tqdm==4.70.1


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 114.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 96.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 17.1 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.6 which is incompatible.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but 

## 2. Select a GPU runtime and mount Drive

In Colab, choose **Runtime → Change runtime type → T4 GPU** (or another available NVIDIA GPU). This cell stops if CUDA is unavailable instead of training accidentally on CPU.

In [3]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU unavailable. In Colab select Runtime > Change runtime type > GPU, then reconnect.")
DEVICE = torch.device("cuda:0")
GPU_NAME = torch.cuda.get_device_name(DEVICE)
print(f"Using {GPU_NAME} ({DEVICE}); PyTorch {torch.__version__}; CUDA runtime {torch.version.cuda}")


Using Tesla T4 (cuda:0); PyTorch 2.11.0+cu130; CUDA runtime 13.0


In [8]:
from google.colab import drive
drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Load the experiment config and prepared Arrow data

The config on Drive is the source of truth for model/qrels revisions, batch size, learning rate, epochs, patience, and output directory. This notebook overrides only machine-specific paths. It copies Arrow data to Colab’s local disk for faster reads; all artifacts remain on Drive.

In [10]:
from pathlib import Path
import shutil
import yaml

DRIVE_PROJECT = Path("/content/drive/MyDrive/fiqa-reranker-drive")
CONFIG_PATH = DRIVE_PROJECT / "configs/step3_easy.yaml"
DRIVE_TOKENIZED = DRIVE_PROJECT / "data/tokenized/fiqa_easy_minilm_l6_v2"
LOCAL_TOKENIZED = Path("/content/fiqa_easy_minilm_l6_v2")
OUTPUT_DIR = DRIVE_PROJECT / "outputs/colab_step3"

for required in (CONFIG_PATH, DRIVE_TOKENIZED / "train", DRIVE_TOKENIZED / "validation", DRIVE_TOKENIZED / "test"):
    if not required.exists():
        raise FileNotFoundError(f"Missing {required}. Copy configs/step3.yaml and the tokenized split folders to {DRIVE_PROJECT}.")

with CONFIG_PATH.open(encoding="utf-8") as f:
    CFG = yaml.safe_load(f)
MODEL_CFG = CFG["model"]
DATA_CFG = CFG["data"]
TRAIN_CFG = CFG["training"]
SEED = int(CFG["seed"])
print(f"Configured batch={TRAIN_CFG['batch_size']}, max_epochs={TRAIN_CFG['max_epochs']}, patience={TRAIN_CFG['patience']}")

if LOCAL_TOKENIZED.exists():
    shutil.rmtree(LOCAL_TOKENIZED)
shutil.copytree(DRIVE_TOKENIZED, LOCAL_TOKENIZED)
print(f"Arrow datasets ready at {LOCAL_TOKENIZED}; outputs will go to {OUTPUT_DIR}")


Configured batch=32, max_epochs=2, patience=2
Arrow datasets ready at /content/fiqa_easy_minilm_l6_v2; outputs will go to /content/drive/MyDrive/fiqa-reranker-drive/outputs/colab_step3


## 4. Reproducibility and schema checks

Training labels must be present and binary. Evaluation rows may have null labels; those rows remain candidates for scoring and are never used in the loss.

In [11]:
import random
import pandas as pd
import numpy as np
from datasets import load_from_disk

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

splits = {name: load_from_disk(str(LOCAL_TOKENIZED / name)) for name in ("train", "validation", "test")}
for name, ds in splits.items():
    required = {"input_ids", "attention_mask", "query_id", "doc_id", "label", "is_judged", "relevance_score"}
    missing = required - set(ds.column_names)
    if missing:
        raise ValueError(f"{name} missing columns: {sorted(missing)}")
    print(f"{name}: {len(ds):,} rows")

train_ds = splits["train"].filter(lambda row: row["label"] is not None, desc="Keep labeled training pairs")
labels = set(float(x) for x in train_ds["label"])
if not labels.issubset({0.0, 1.0}):
    raise ValueError(f"Training labels are not binary: {labels}")
if len(train_ds) == 0:
    raise ValueError("No labeled training examples")
for name in ("validation", "test"):
    ds = splits[name]
    if any(int(judged) == 0 and label is not None for judged, label in zip(ds["is_judged"], ds["label"], strict=True)):
        raise ValueError(f"{name} has labels on unjudged rows")
query_ids_by_split = {name: set(map(str, ds["query_id"])) for name, ds in splits.items()}
for left, right in (("train", "validation"), ("train", "test"), ("validation", "test")):
    overlap = query_ids_by_split[left] & query_ids_by_split[right]
    if overlap:
        raise ValueError(f"Query ID leakage between {left} and {right}: {len(overlap)} IDs")
print(f"Training label counts: {pd.Series(train_ds['label']).value_counts().to_dict()}")
print(f"Labeled training rows: {len(train_ds):,}; CUDA device: {GPU_NAME}")


train: 70,655 rows
validation: 50,000 rows
test: 64,754 rows


Keep labeled training pairs:   0%|          | 0/70655 [00:00<?, ? examples/s]

Training label counts: {0.0: 56524, 1.0: 14131}
Labeled training rows: 70,655; CUDA device: Tesla T4


## 5. Load the pinned model and tokenizer

The model is downloaded from the immutable revision in the config and cached on the Colab VM. This notebook starts from the original pretrained weights, matching the local training script.

In [12]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_NAME = MODEL_CFG["name"]
MODEL_REVISION = MODEL_CFG["revision"]
MODEL_CACHE = Path("/content/model")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION, cache_dir=MODEL_CACHE)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, revision=MODEL_REVISION, cache_dir=MODEL_CACHE).to(DEVICE)
print(f"Loaded {MODEL_NAME}@{MODEL_REVISION}; max tokenized pair length={max(len(x) for x in train_ds['input_ids']):,}")


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loaded cross-encoder/ms-marco-MiniLM-L6-v2@233902d25c440f23af6f7d6e94d2946bac0bee0a; max tokenized pair length=256


## 6. Dynamic padding, optimizer, and schedule

Each batch is padded only to that batch’s longest sequence. AdamW excludes bias and LayerNorm weights from decay. The linear scheduler warms up for the configured fraction of optimizer updates.

In [13]:
from torch.utils.data import DataLoader
from transformers import get_linear_schedule_with_warmup

def collate_train(rows):
    keys = ("input_ids", "attention_mask", "token_type_ids")
    model_inputs = [{key: row[key] for key in keys if key in row and row[key] is not None} for row in rows]
    padded = tokenizer.pad(model_inputs, padding=True, return_tensors="pt")
    y = torch.tensor([float(row["label"]) for row in rows], dtype=torch.float32)
    return padded, y

generator = torch.Generator().manual_seed(SEED)
batch_size = int(TRAIN_CFG["batch_size"])
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, generator=generator, collate_fn=collate_train, num_workers=int(TRAIN_CFG.get("num_workers", 0)))

no_decay = ("bias", "LayerNorm.weight", "layer_norm.weight", "layernorm.weight")
decay_params, no_decay_params = [], []
for name, param in model.named_parameters():
    if param.requires_grad:
        (no_decay_params if any(key in name for key in no_decay) else decay_params).append(param)
optimizer = torch.optim.AdamW(
    [{"params": decay_params, "weight_decay": float(TRAIN_CFG["weight_decay"])}, {"params": no_decay_params, "weight_decay": 0.0}],
    lr=float(TRAIN_CFG["learning_rate"]),
)
max_epochs = int(TRAIN_CFG["max_epochs"])
total_steps = len(train_loader) * max_epochs
warmup_steps = int(total_steps * float(TRAIN_CFG["warmup_ratio"]))
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
amp_enabled = bool(TRAIN_CFG.get("mixed_precision", True))
scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)
criterion = torch.nn.BCEWithLogitsLoss()
print(f"{len(train_loader):,} optimizer updates/epoch; {total_steps:,} planned updates; {warmup_steps:,} warmup updates; AMP={amp_enabled}")


2,208 optimizer updates/epoch; 4,416 planned updates; 441 warmup updates; AMP=True


## 7. Load the original validation qrels and metric helpers

Qrels come from the pinned official FiQA qrels dataset. nDCG@10 uses graded qrels gains and an ideal ranking over all judged positive qrels. MRR@10 uses the first positive qrel in the scored candidate list. Unjudged candidates retain their rank positions but are not assigned explicit labels.

In [16]:
import math
import pandas as pd
from datasets import load_dataset

QRELS_CFG = DATA_CFG["qrels"]
def load_qrels(split):
    hf_split = QRELS_CFG["split_names"][split]
    frame = load_dataset(QRELS_CFG["name"], split=hf_split, revision=QRELS_CFG["revision"]).to_pandas()
    needed = {"query-id", "corpus-id", "score"}
    if not needed.issubset(frame.columns):
        raise ValueError(f"Official qrels missing: {sorted(needed - set(frame.columns))}")
    frame = frame.rename(columns={"query-id": "query_id", "corpus-id": "doc_id"})
    frame["query_id"] = frame.query_id.astype(str)
    frame["doc_id"] = frame.doc_id.astype(str)
    frame["score"] = pd.to_numeric(frame.score, errors="raise")
    if frame.duplicated(["query_id", "doc_id"]).any():
        if (frame.groupby(["query_id", "doc_id"]).score.nunique() > 1).any():
            raise ValueError("Conflicting official qrels")
        frame = frame.drop_duplicates(["query_id", "doc_id"])
    result = {}
    for row in frame.itertuples(index=False):
        result.setdefault(row.query_id, {})[row.doc_id] = float(row.score)
    return result

validation_qrels = load_qrels("validation")
print(f"Loaded original validation qrels for {len(validation_qrels):,} queries")


README.md:   0%|          | 0.00/14.0k [00:00<?, ?B/s]

train.tsv:   0%|          | 0.00/210k [00:00<?, ?B/s]

dev.tsv:   0%|          | 0.00/18.3k [00:00<?, ?B/s]

test.tsv:   0%|          | 0.00/25.3k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Loaded original validation qrels for 500 queries


## 8. Train and validate each epoch

The loop mirrors the script: explicit BCE-with-logits loss, float labels, dynamic padding, gradient clipping, linear warmup/decay, epoch checkpoints, validation nDCG-based best selection, and patience-based early stopping. Test qrels are not loaded in this cell.

In [18]:
import math

def ranking_metrics(predictions, qrels, k=10):
    grouped = {str(qid): group.sort_values("score", ascending=False, kind="stable") for qid, group in predictions.groupby("query_id", sort=False)}
    ndcgs, reciprocal_ranks = [], []
    for qid, judgments in qrels.items():
        if not any(value > 0 for value in judgments.values()):
            continue
        group = grouped.get(str(qid))
        ranked_ids = [] if group is None else group.doc_id.astype(str).tolist()[:k]
        dcg = sum((2 ** max(0.0, judgments[doc_id]) - 1) / math.log2(rank + 2) for rank, doc_id in enumerate(ranked_ids) if doc_id in judgments)
        ideal = sorted((max(0.0, value) for value in judgments.values()), reverse=True)[:k]
        idcg = sum((2 ** gain - 1) / math.log2(rank + 2) for rank, gain in enumerate(ideal))
        ndcgs.append(dcg / idcg if idcg else 0.0)
        reciprocal_ranks.append(next((1.0 / (rank + 1) for rank, doc_id in enumerate(ranked_ids) if judgments.get(doc_id, 0) > 0), 0.0))
    return {"ndcg@10": float(np.mean(ndcgs)) if ndcgs else 0.0, "mrr@10": float(np.mean(reciprocal_ranks)) if reciprocal_ranks else 0.0, "queries": len(ndcgs), "k": k}


from tqdm.auto import tqdm
import json
import importlib.metadata

def score_candidates(model, dataset, desc):
    model.eval()
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=lambda rows: (
        tokenizer.pad([{key: row[key] for key in ("input_ids", "attention_mask", "token_type_ids") if key in row and row[key] is not None} for row in rows], padding=True, return_tensors="pt"),
        [str(row["query_id"]) for row in rows], [str(row["doc_id"]) for row in rows],
    ))
    records = []
    with torch.inference_mode():
        for inputs, query_ids, doc_ids in tqdm(loader, desc=desc, unit="batch"):
            inputs = {key: value.to(DEVICE) for key, value in inputs.items()}
            logits = model(**inputs).logits.reshape(-1).float().cpu().tolist()
            records.extend({"query_id": qid, "doc_id": did, "score": float(score)} for qid, did, score in zip(query_ids, doc_ids, logits, strict=True))
    return pd.DataFrame(records, columns=["query_id", "doc_id", "score"])

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
resolved_cfg = dict(CFG)
resolved_cfg["device"] = str(DEVICE)
resolved_cfg["gpu_name"] = GPU_NAME
resolved_cfg["data"] = dict(DATA_CFG, tokenized_dir=str(LOCAL_TOKENIZED))
resolved_cfg["training"] = dict(TRAIN_CFG, output_dir=str(OUTPUT_DIR))
with (OUTPUT_DIR / "resolved_config.yaml").open("w", encoding="utf-8") as f:
    yaml.safe_dump(resolved_cfg, f, sort_keys=False)

history = []
best_ndcg, best_epoch, stale = -float("inf"), 0, 0
patience = int(TRAIN_CFG["patience"])
for epoch in range(1, max_epochs + 1):
    model.train()
    losses = []
    progress = tqdm(train_loader, desc=f"Epoch {epoch}/{max_epochs}: train", unit="batch")
    for inputs, labels in progress:
        inputs = {key: value.to(DEVICE) for key, value in inputs.items()}
        labels = labels.to(DEVICE, dtype=torch.float32)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp_enabled):
            logits = model(**inputs).logits.reshape(-1).float()
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), float(TRAIN_CFG["gradient_clip_norm"]))
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        losses.append(float(loss.detach().cpu()))
        progress.set_postfix(bce=f"{losses[-1]:.4f}", lr=f"{scheduler.get_last_lr()[0]:.2e}")

    print(f"Epoch {epoch}: scoring validation candidates")
    val_predictions = score_candidates(model, splits["validation"], f"Epoch {epoch}: validation")
    val_predictions.to_csv(OUTPUT_DIR / f"validation_epoch_{epoch:02d}_scores.tsv", sep="\t", index=False)
    metrics = ranking_metrics(val_predictions, validation_qrels, k=10)
    row = {"epoch": epoch, "train_bce_loss": float(np.mean(losses)), "validation_ndcg@10": metrics["ndcg@10"], "validation_mrr@10": metrics["mrr@10"], "validation_queries": metrics["queries"], "learning_rate": scheduler.get_last_lr()[0]}
    history.append(row)
    if bool(TRAIN_CFG.get("save_every_epoch", True)):
        epoch_dir = OUTPUT_DIR / f"checkpoint_epoch_{epoch:02d}"
        model.save_pretrained(epoch_dir, safe_serialization=True)
        tokenizer.save_pretrained(epoch_dir)
    if float(metrics["ndcg@10"]) > best_ndcg:
        best_ndcg, best_epoch, stale = float(metrics["ndcg@10"]), epoch, 0
        model.save_pretrained(OUTPUT_DIR / "best_model", safe_serialization=True)
        tokenizer.save_pretrained(OUTPUT_DIR / "best_model")
    else:
        stale += 1
    with (OUTPUT_DIR / "training_history.json").open("w", encoding="utf-8") as f:
        json.dump({"epochs": history, "best_epoch": best_epoch, "best_validation_ndcg@10": best_ndcg}, f, indent=2)
        f.write("\n")
    print(f"Epoch {epoch}: train BCE={row['train_bce_loss']:.6f}; validation nDCG@10={metrics['ndcg@10']:.6f}; MRR@10={metrics['mrr@10']:.6f}; stale={stale}/{patience}")
    if stale >= patience:
        print("Early stopping: validation nDCG@10 did not improve.")
        break


Epoch 1/2: train:   0%|          | 0/2208 [00:00<?, ?batch/s]

Epoch 1: scoring validation candidates


Epoch 1: validation:   0%|          | 0/1563 [00:00<?, ?batch/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Epoch 2/2: train:   0%|          | 0/2208 [00:00<?, ?batch/s]

Epoch 2: scoring validation candidates


Epoch 2: validation:   0%|          | 0/1563 [00:00<?, ?batch/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 2: train BCE=0.076396; validation nDCG@10=0.353186; MRR@10=0.435839; stale=1/2


## 9. Restore the best checkpoint and evaluate test once

This is the only cell that reads test qrels. Do not change model settings based on these results; use validation for any future tuning.

In [19]:
best_model = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR / "best_model", local_files_only=True).to(DEVICE)
test_predictions = score_candidates(best_model, splits["test"], "Best checkpoint: test scoring")
test_predictions.to_csv(OUTPUT_DIR / "test_scores.tsv", sep="\t", index=False)
test_qrels = load_qrels("test")
test_metrics = ranking_metrics(test_predictions, test_qrels, k=10)
test_report = {**test_metrics, "best_epoch": best_epoch, "checkpoint": "best_model", "note": "One final test evaluation after validation-based selection; do not tune on test."}
with (OUTPUT_DIR / "test_metrics.json").open("w", encoding="utf-8") as f:
    json.dump(test_report, f, indent=2)
    f.write("\n")
metadata = {
    "model": MODEL_CFG, "seed": SEED, "device": str(DEVICE), "gpu_name": GPU_NAME,
    "best_epoch": best_epoch, "test_metrics": test_metrics,
    "package_versions": {name: importlib.metadata.version(name) for name in ("torch", "transformers", "datasets", "pandas")},
}
with (OUTPUT_DIR / "run_metadata.json").open("w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)
    f.write("\n")
print(f"Final test nDCG@10={test_metrics['ndcg@10']:.6f}; MRR@10={test_metrics['mrr@10']:.6f}")
print(f"All outputs saved to {OUTPUT_DIR}")


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Best checkpoint: test scoring:   0%|          | 0/2024 [00:00<?, ?batch/s]

Final test nDCG@10=0.337821; MRR@10=0.413795
All outputs saved to /content/drive/MyDrive/fiqa-reranker-drive/outputs/colab_step3


## Optional: one-batch overfit sanity check

This cell is disabled by default. Set `RUN_OVERFIT_DIAGNOSTIC = True` and run it to initialize a separate model from the original pinned weights, repeatedly optimize one labeled batch, record its loss, then verify the original weights were restored. It does not alter the model used by the normal training loop.

In [ ]:
# Optional: execute this cell separately before the main training cell.
RUN_OVERFIT_DIAGNOSTIC = False
if RUN_OVERFIT_DIAGNOSTIC:
    diagnostic_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, revision=MODEL_REVISION, cache_dir=MODEL_CACHE).to(DEVICE)
    original_state = {key: value.detach().cpu().clone() for key, value in diagnostic_model.state_dict().items()}
    overfit_rows = next(iter(DataLoader(train_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_train)))
    overfit_inputs = {key: value.to(DEVICE) for key, value in overfit_rows[0].items()}
    overfit_labels = overfit_rows[1].to(DEVICE)
    overfit_optimizer = torch.optim.AdamW(diagnostic_model.parameters(), lr=float(TRAIN_CFG["overfit"]["learning_rate"]), weight_decay=0.0)
    overfit_trace = []
    for step in tqdm(range(1, int(TRAIN_CFG["overfit"]["steps"]) + 1), desc="One-batch overfit", unit="step"):
        diagnostic_model.train()
        overfit_optimizer.zero_grad(set_to_none=True)
        logits = diagnostic_model(**overfit_inputs).logits.reshape(-1).float()
        loss = criterion(logits, overfit_labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(diagnostic_model.parameters(), float(TRAIN_CFG["gradient_clip_norm"]))
        overfit_optimizer.step()
        overfit_trace.append({"step": step, "bce_loss": float(loss.detach().cpu())})
    diagnostic_model.load_state_dict(original_state)
    reloaded_original = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, revision=MODEL_REVISION, cache_dir=MODEL_CACHE).to(DEVICE)
    assert all(torch.equal(value.detach().cpu(), reloaded_original.state_dict()[key].detach().cpu()) for key, value in diagnostic_model.state_dict().items())
    pd.DataFrame(overfit_trace).to_csv(OUTPUT_DIR / "overfit_loss.tsv", sep="\t", index=False)
    del diagnostic_model, reloaded_original
    print(f"Original pretrained weights reloaded and verified. Loss trace saved under {OUTPUT_DIR}.")


## Exercise

After a run, compare training BCE and validation nDCG by epoch. Which is a better signal for choosing the model? Why should the test metric not drive another hyperparameter change? Keep any future experiment in a new output directory so its config and checkpoint remain distinct.